# Vígil.ia — Campeão + vídeos em 1280 DE VERDADE (teste de resolução)

**Por que este notebook existe:** a rodada anterior treinou com `imgsz=1280` mas o
dataset estava **assado em 640** (upscale puro — 4× o custo, zero detalhe novo) e a
avaliação rodou em 640 (mismatch treino/inferência). O resultado ficou incomparável.

Aqui tudo é 1280 de ponta a ponta:
1. **v3 reconstruído em 1280** das fotos originais (letterbox, labels e sintético).
2. **Frames de vídeo letterboxados em 1280** (segmentação Otsu v1, com corte da
   faixa da máquina e filtros anti-lixo).
3. **Treino em 1280** (batch=-1, auto no limite da VRAM) partindo do campeão.
4. **Avaliação casada**: cada modelo roda na resolução em que treinou —
   campeão@640 · video_v1 (640, se existir)@640 · **video_v1_1280@1280** — com mAP
   no val correspondente (640 ou HD) e latência por frame.

Assim a diferença 640→1280 vira **uma variável isolada**.

> Custo: dataset HD ~20 min + treino ~1,5–2,5h na L4 (batch auto).
> ⚠️ Vídeo só de intacto = eixo 1 mede só falso-alarme; e a avaliação nos vídeos de
> treino é contaminada (o modelo os viu) — o juiz limpo é o `teste_soja` e o mAP.

## 0. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU!'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Caminhos + config

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

CHAMPION_PT = '/content/drive/MyDrive/soja_yolo11n_baseline.pt'
assert os.path.exists(CHAMPION_PT), f'campeão não encontrado: {CHAMPION_PT}'

REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]
VAL_ROOT = '/content/drive/MyDrive/Vídeos para treino/Treino'
assert os.path.isdir(VAL_ROOT), f'VAL_ROOT não existe: {VAL_ROOT}'

SIZE = 1280          # <- resolução do experimento, de ponta a ponta
FRAME_STRIDE = 5
VIDEO_WEIGHT = 1
print('campeão:', CHAMPION_PT, '| SIZE =', SIZE)

## 2. Funções de dataset (as mesmas dos outros notebooks)

In [ ]:
import glob, hashlib, unicodedata, cv2, yaml
import numpy as np

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
RNG = np.random.default_rng(42)

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    from collections import Counter
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    return items

def sat_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def motion_blur(img, rng=RNG):
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)

def extract_cutout(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    x, y, bw, bh = cv2.boundingRect(c)
    return img[y:y + bh, x:x + bw], mask[y:y + bh, x:x + bw]

def make_scene(cutouts, rng=RNG, size=640):
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(60, 150)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes

def balance_train(items):
    from collections import defaultdict
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = defaultdict(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    print('balanceado (train):', {NAMES[c]: sum(1 for it in out if it[1] == c) for c in sorted(by)})
    return out + rest

def build_v3(items, out_dir, n_synth=600, blur_frac=0.4):
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cutouts = []
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            skipped += 1; continue
        if sp == 'train':
            cut = extract_cutout(img)
            if cut is not None:
                cutouts.append((cls, cut[0], cut[1]))
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(line)
        kept += 1
        if sp == 'train':
            cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
            kept += 1
    print(f'fotos reais: kept={kept} skipped={skipped} | recortes: {len(cutouts)}')
    assert cutouts, 'Nenhum recorte extraído!'
    synth = 0
    for j in range(n_synth):
        if j % 100 == 0:
            print(f'  cenas {j}/{n_synth}…', flush=True)
        canvas, boxes = make_scene(cutouts)
        if not boxes:
            continue
        if RNG.random() < blur_frac:
            canvas = motion_blur(canvas)
        stem = f'synth_{j:05d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(
            '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}' for c, cx, cy, w, h in boxes))
        synth += 1
    print(f'cenas sintéticas: {synth}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

SPLIT_MAP = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}

def collect_base(base_dir):
    """Acha train/valid/test em qualquer profundidade dentro do dataset 12,5k."""
    items = []
    for root, dirs, _ in os.walk(base_dir):
        for d in list(dirs):
            sp = SPLIT_MAP.get(d.lower())
            if sp is None:
                continue
            split_dir = os.path.join(root, d)
            for folder in sorted(os.listdir(split_dir)):
                cls = class_of(folder)
                if cls is None:
                    continue
                for p_ in glob.glob(os.path.join(split_dir, folder, '*')):
                    if p_.lower().endswith(IMG_EXT):
                        items.append((p_, cls, sp))
            dirs.remove(d)
    from collections import Counter
    print('coletado (base 12,5k):', dict(Counter(sp for _, _, sp in items)))
    return items

def build_base(items, out_dir):
    """Dataset base de detecção: pseudo-rótulo Otsu (1 grão/img, fundo preto),
    sem balanceamento/blur/sintético — idêntico ao estágio base do RT-DETR."""
    assert items, 'Nenhuma imagem do 12,5k coletada!'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 1000 == 0:
            print(f'  base {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = otsu_box(img)
        if box is None:
            skipped += 1; continue
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(
            f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}')
        kept += 1
    print(f'base: kept={kept} skipped={skipped}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

## 3. Dataset v3 em 1280 — reconstruído das fotos originais
Letterbox, labels e cenas sintéticas em `SIZE`. (~15–20 min; ~4× mais disco)

In [ ]:
def build_v3_hd(items, out_dir, size=SIZE, n_synth=600, blur_frac=0.4):
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cutouts = []
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            skipped += 1; continue
        if sp == 'train':
            cut = extract_cutout(img)
            if cut is not None:
                cutouts.append((cls, cut[0], cut[1]))
        lb, s, left, top = letterbox640(img, size=size)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / size
        cy = (cy * h0 * s + top) / size
        ww = (ww * w0 * s) / size
        hh = (hh * h0 * s) / size
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 92])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(line)
        kept += 1
        if sp == 'train':
            cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb),
                        [cv2.IMWRITE_JPEG_QUALITY, 92])
            open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
            kept += 1
    print(f'fotos reais: kept={kept} skipped={skipped} | recortes: {len(cutouts)}')
    assert cutouts, 'Nenhum recorte extraído!'
    for j in range(n_synth):
        if j % 100 == 0:
            print(f'  cenas {j}/{n_synth}…', flush=True)
        canvas, boxes = make_scene(cutouts, size=size)
        if not boxes:
            continue
        if RNG.random() < blur_frac:
            canvas = motion_blur(canvas)
        stem = f'synth_{j:05d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 92])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(
            '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}' for c, cx, cy, w, h in boxes))
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

HD_YAML = f'/content/soja_det_hd{SIZE}/data.yaml'
if not os.path.exists(HD_YAML):
    HD_YAML = build_v3_hd(collect_real(REAL_SRCS), f'/content/soja_det_hd{SIZE}')
print('dataset v3 HD:', HD_YAML)

# v3 padrão 640 — só p/ avaliar os modelos 640 no mAP (não treina nada aqui)
V3_YAML = '/content/soja_det_v3/data.yaml'
if not os.path.exists(V3_YAML):
    V3_YAML = build_v3(collect_real(REAL_SRCS), '/content/soja_det_v3')
print('dataset v3 640:', V3_YAML)

## 4. Segmentação Otsu (v1) + dataset de VÍDEO em 1280
Frames cortados acima da faixa da máquina, letterbox em `SIZE`, labels normalizados
por `SIZE`. Split por vídeo (ou temporal com folga).

In [ ]:
VIDEO_EXT = ('.mp4', '.mov', '.avi', '.mkv')
ROI_BOTTOM = 0.15
MIN_SAT    = 40

def video_frames(path, stride=FRAME_STRIDE):
    cap = cv2.VideoCapture(path)
    k = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if k % stride == 0:
            yield k, frame[:int(frame.shape[0] * (1 - ROI_BOTTOM))]
        k += 1
    cap.release()

def multi_boxes(img, min_frac=0.0008, max_frac=0.05):
    """Otsu v1: saturação -> Otsu global + filtros anti-lixo. Sem modelo!"""
    h, w = img.shape[:2]
    S = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 1]
    blur = cv2.GaussianBlur(S, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in cnts:
        a = cv2.contourArea(c)
        if not (min_frac * h * w <= a <= max_frac * h * w):
            continue
        x, y, bw, bh = cv2.boundingRect(c)
        if bw / max(bh, 1) > 3 or bh / max(bw, 1) > 3:
            continue
        if a < 0.4 * bw * bh:
            continue
        if S[y:y+bh, x:x+bw].mean() < MIN_SAT:
            continue
        pad = int(0.04 * min(bw, bh)) + 2
        x1, y1 = max(0, x - pad), max(0, y - pad)
        x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
        boxes.append((((x1+x2)/2)/w, ((y1+y2)/2)/h, (x2-x1)/w, (y2-y1)/h))
    return boxes

VID = f'/content/soja_video_hd{SIZE}'
import shutil as _sh
_sh.rmtree(VID, ignore_errors=True)
for sp in ('train', 'val'):
    os.makedirs(f'{VID}/images/{sp}', exist_ok=True)
    os.makedirs(f'{VID}/labels/{sp}', exist_ok=True)

stats = {}
for entry in sorted(os.listdir(VAL_ROOT)):
    sub = os.path.join(VAL_ROOT, entry)
    if not os.path.isdir(sub):
        continue
    cls = class_of(entry)
    if cls is None:
        continue
    vids = [os.path.join(sub, f) for f in sorted(os.listdir(sub))
            if f.lower().endswith(VIDEO_EXT)]
    if not vids:
        continue
    val_video = vids[-1] if len(vids) >= 2 else None
    wrote = {'train': 0, 'val': 0}
    for vi, vp in enumerate(vids):
        frames = list(video_frames(vp))
        if val_video is None:
            cut = int(0.85 * len(frames))
            split_of = lambda i: ('train' if i < cut else
                                  None if i < cut + 30 // FRAME_STRIDE else 'val')
        else:
            split_of = lambda i: 'val' if vp == val_video else 'train'
        for i, (k, frame) in enumerate(frames):
            sp = split_of(i)
            if sp is None:
                continue
            boxes = multi_boxes(frame)
            if not (3 <= len(boxes) <= 80):
                continue
            lb, s, left, top = letterbox640(frame, size=SIZE)   # letterbox em 1280!
            H0, W0 = frame.shape[:2]
            lines = []
            for cx, cy, ww, hh in boxes:
                cx2 = (cx * W0 * s + left) / SIZE
                cy2 = (cy * H0 * s + top) / SIZE
                lines.append(f'{cls} {cx2:.6f} {cy2:.6f} {ww*W0*s/SIZE:.6f} {hh*H0*s/SIZE:.6f}')
            stem = f'{NAMES[cls]}_{vi}_{k:05d}'
            cv2.imwrite(f'{VID}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 92])
            open(f'{VID}/labels/{sp}/{stem}.txt', 'w').write('\n'.join(lines))
            wrote[sp] += 1
    stats[NAMES[cls]] = wrote
    print(f'{entry} -> {NAMES[cls]}: {wrote}')

assert any(v['train'] for v in stats.values()), 'Nenhum frame de vídeo escrito!'
print('\ndataset de vídeo HD pronto:', stats)

## 5. CONFIRA O DATASET ANTES DE TREINAR ⚠️
Todo grão com caixa, justas, sem lixo, sem fundir grãos. Ruim? Ajuste e refaça a célula 4.

In [ ]:
import glob, random
import matplotlib.pyplot as plt

samples = random.sample(glob.glob(f'{VID}/images/train/*.jpg'),
                        min(6, len(glob.glob(f'{VID}/images/train/*.jpg'))))
plt.figure(figsize=(15, 10))
for i, ip in enumerate(samples):
    img = cv2.imread(ip)
    lp = ip.replace('/images/', '/labels/').replace('.jpg', '.txt')
    n = 0
    for ln in open(lp):
        c, cx, cy, ww, hh = ln.split()
        cx, cy, ww, hh = float(cx)*SIZE, float(cy)*SIZE, float(ww)*SIZE, float(hh)*SIZE
        cv2.rectangle(img, (int(cx-ww/2), int(cy-hh/2)), (int(cx+ww/2), int(cy+hh/2)),
                      (0, 255, 0), 3)
        n += 1
    ax = plt.subplot(2, 3, i + 1)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{os.path.basename(ip)} — {n} caixas', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Treino em 1280 — batch=-1 (auto no limite da VRAM)
Mix v3 HD + vídeo HD, partindo do campeão, lr 5e-4.

In [ ]:
import yaml
from ultralytics import YOLO
import shutil

MIX_YAML = '/content/soja_mix_hd.yaml'
train_dirs = [f'/content/soja_det_hd{SIZE}/images/train'] + [f'{VID}/images/train'] * VIDEO_WEIGHT
yaml.safe_dump({'train': train_dirs,
                'val': f'/content/soja_det_hd{SIZE}/images/val',
                'names': {i: n for i, n in enumerate(NAMES)}},
               open(MIX_YAML, 'w'), sort_keys=False, allow_unicode=True)
print('mix:', train_dirs)

DST = '/content/drive/MyDrive/soja_yolo11n_video_v1_1280.pt'
if os.path.exists(DST):
    print('já treinado:', DST)
else:
    m = YOLO(CHAMPION_PT)
    m.train(data=MIX_YAML, epochs=40, imgsz=SIZE, batch=-1, device=0, seed=42,
            optimizer='AdamW', lr0=5e-4, patience=15, close_mosaic=8,
            cache=False, workers=8,
            mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
            fliplr=0.5, flipud=0.5,
            project='runs_campeao', name='11n_video_v1_1280', exist_ok=True)
    shutil.copy(str(m.trainer.best), DST)
    print('salvo:', DST)

## 7. Placar — cada modelo na SUA resolução
campeão@640 · video_v1 (640, se existir)@640 · video_v1_1280@1280.
Vídeo (por detecção) + mAP no val correspondente + latência. A diferença entre
video_v1@640 e video_v1_1280@1280 é o **efeito puro da resolução** (mesmo dado).

In [ ]:
import time
from collections import Counter
from ultralytics import YOLO

CASOS = [('campeão (640)',        CHAMPION_PT,                                          640,  V3_YAML),
         ('video_v1 (640)',       '/content/drive/MyDrive/soja_yolo11n_video_v1.pt',    640,  V3_YAML),
         ('video_v1_1280 (1280)', '/content/drive/MyDrive/soja_yolo11n_video_v1_1280.pt', SIZE, HD_YAML)]

by_class = {}
for entry in sorted(os.listdir(VAL_ROOT)):
    sub = os.path.join(VAL_ROOT, entry)
    if not os.path.isdir(sub):
        continue
    c = class_of(entry)
    if c is None:
        continue
    vids = [os.path.join(sub, f) for f in sorted(os.listdir(sub))
            if f.lower().endswith(VIDEO_EXT)]
    if vids:
        by_class[NAMES[c]] = vids

def video_eval(pt, imgsz, vid_stride=5):
    model = YOLO(pt)
    tot, cert, nfr, t = Counter(), Counter(), 0, 0.0
    for true_cls, vids in by_class.items():
        for vid in vids:
            t0 = time.time()
            for r in model.predict(source=vid, imgsz=imgsz, conf=0.35, iou=0.5,
                                   agnostic_nms=True, vid_stride=vid_stride,
                                   stream=True, verbose=False):
                nfr += 1
                for c in r.boxes.cls.int().tolist():
                    tot[true_cls] += 1
                    if NAMES[c] == true_cls:
                        cert[true_cls] += 1
            t += time.time() - t0
    return tot, cert, 1000 * t / max(nfr, 1)

print('=== eixo 1: vídeo (por detecção) — ATENÇÃO: vídeos de treino = contaminado p/ os video_v1 ===')
print(f'{"caso":24s} {"acurácia":>9s} {"det":>6s} {"ms/frame":>9s}')
for tag, pt, sz, _ in CASOS:
    if not os.path.exists(pt):
        print(f'{tag:24s}  (ausente — pulado)'); continue
    tot, cert, ms = video_eval(pt, sz)
    n, ok = sum(tot.values()), sum(cert.values())
    print(f'{tag:24s} {100*ok/max(n,1):8.1f}% {n:6d} {ms:9.1f}')

print('\n=== eixo 2: mAP no val correspondente (o juiz limpo) ===')
for tag, pt, sz, dyaml in CASOS:
    if not os.path.exists(pt):
        continue
    r = YOLO(pt).val(data=dyaml, split='val', imgsz=sz, device=0, verbose=False)
    per = {NAMES[int(i)]: f'{ap:.2f}' for i, ap in
           zip(r.box.ap_class_index, r.box.maps[r.box.ap_class_index])}
    print(f'  {tag:24s}: mAP50={r.box.map50:.3f}  por classe: {per}')

print('\nLeitura: video_v1(640) vs video_v1_1280(1280) = efeito PURO da resolução')
print('(mesmo dado, mesmo treino). Ganho < ~3pp não paga os ~4x de latência.')

## Como fechar
- **video_v1_1280 ≫ video_v1 (640)** nos dois eixos → resolução importa; considere
  1280 (ou o meio-termo 960) no pipeline — pagando a latência ~4×.
- **≈ iguais** → resolução era irrelevante; o que estava distorcendo a rodada
  anterior era o mismatch (treino 1280 sobre dataset 640, avaliado em 640).
- Nos dois casos, o critério de promoção continua: mAP dos **defeitos** (broken/
  skin-damaged) não pode desabar vs o campeão. E o juiz sem contaminação é o
  `teste_soja` + mAP, não os vídeos de treino.